In [1]:
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from llama_cpp import Llama
import pandas as pd
from tqdm import tqdm
import re
from sklearn.model_selection import train_test_split
import os

In [32]:
file_path = "./cleaned_news_dataset_w_derived_columns.csv"
df = pd.read_csv(file_path)

In [34]:
df.head()

,title,text,subject,date,label,date_clean,parsed_date,exclamation_count_title,question_count_title,quotes_count_title,...,ellipsis_count_text,hashtag_count_text,mention_count_text,noun_ratio_text,verb_ratio_text,adj_ratio_text,sentiment_polarity_text,sentiment_subjectivity_text,capitalization_ratio_text,html_link_count_text
0,"WaPo Editorial Goes Full Racist, Wants To ‘We...",Right-wing ideologues are at it again with vot...,worldnews,"May 22, 2016",1,"May 22, 2016",2016-05-22,0,0,0,...,0,0,0,0.273279,0.159919,0.103239,0.018619,0.439495,0.018219,0
1,Trump sets out strong trade message at Asia-Pa...,U.S. President Donald Trump set out a strong m...,worldnews,"November 10, 2017",0,"November 10, 2017",2017-11-10,0,0,0,...,0,0,0,0.333333,0.108696,0.086957,0.147619,0.654762,0.021739,0
2,"Clinton to resume campaigning on Thursday, 'de...",Democratic presidential nominee Hillary Clinto...,politicsNews,"September 13, 2016",0,"September 13, 2016",2016-09-13,0,0,0,...,0,0,0,0.335052,0.185567,0.051546,0.093636,0.315455,0.020619,0
3,FBI: Clinton Foundation investigation will lea...,21st Century Wire says Yesterday we learned th...,worldnews,"November 4, 2016",1,"November 4, 2016",2016-11-04,0,0,0,...,0,0,1,0.333333,0.158824,0.086275,0.150740,0.468074,0.047059,0
4,North Carolina Just Came Up With The Most Lud...,"North Carolina, which is still butthurt over t...",worldnews,"September 12, 2016",1,"September 12, 2016",2016-09-12,0,0,0,...,0,0,0,0.276744,0.155814,0.081395,0.154948,0.432552,0.006977,0


In [35]:
def prepare_data_splits(df, test_size=0.2, val_size=0.2):
    train_val_df, test_df = train_test_split(df, test_size=test_size, stratify=df['label'], random_state=42)
    train_df, val_df = train_test_split(train_val_df, test_size=val_size, stratify=train_val_df['label'], random_state=42)
    return train_df, val_df, test_df

In [36]:
_, _, test_df = prepare_data_splits(df)

In [37]:
len(test_df)

7680

In [ ]:
api_token = "your-API-token"

In [39]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B", use_auth_token=api_token)

def count_tokens(prompt):
    return len(tokenizer.encode(prompt, truncation=False))

/home/noaoh/.conda/envs/my_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/noaoh/.conda/envs/my_env/lib/python3.10/site-packages/transformers/models/auto/tokenization_auto.py:862: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


In [40]:
prompt_template_title = PromptTemplate.from_template("""
You are a news classification assistant. Classify each news as Real or Fake.

Examples:
# Example 1
Title: From prairie to the White House: Inside a Tribe's quest to stop a pipeline
Label: Real

# Example 2
Title: Michelle Obama Just BLASTED Conservatives For Their Racist Lies About Our President (VIDEO)
Label: Fake                                                  

Now classify: 
Title: "{news_title}"
The label of this news is:
"""
)

prompt_template_titleVtext = PromptTemplate.from_template("""
You are a news classification assistant. Classify each news as Real or Fake.

Examples:
# Example 1
Title: From prairie to the White House: Inside a Tribe's quest to stop a pipeline
Text: Three days after guard dogs attacked Native Americans protesting an oil pipeline project in North Dakota in early September, an unprecedented event took place at the White House. Brian Cladoosby, president of the National Congress of American Indians, which represents more than 500 tribes, spoke to nearly a dozen of President Barack Obama’s Cabinet-level advisers at a September 6 meeting of the White House’s three-year-old Native American Affairs Council.  It was the first time a tribal leader addressed a session of the council, and Cladoosby was invited in his role as the Indian Congress’ leader. Cladoosby, a Swinomish Indian from Washington state, spoke twice at the one-hour roundtable. He told Reuters he praised the Obama administration in his opening statement for its track record on Native American issues such as pushing to reform the Indian Health Service. But when Cladoosby gave his closing speech, he delivered an impassioned request to his audience: stand with Native Americans who have united with the Standing Rock Sioux tribe and block construction of the Dakota Access Pipeline, a 1,100 mile conduit to get oil from North Dakota to Illinois.  That plea marked one of the previously unreported turning points in a drama that played out since February and culminated September 9 with an about face by the U.S. government, from giving the pipeline a green light to backing a request from North Dakota’s Standing Rock Sioux to halt construction of the pipeline.  The tribe fears sacred sites could be destroyed during the line’s construction and that a future oil spill would pollute its drinking water. This month’s win for the tribe, which could be reversed by regulators, is a rare instance of protests resulting in quick federal action and the triumph of an unusual alliance between environmentalists and Native Americans, who both say they were emboldened by the defeat of the Keystone XL pipeline last fall. It also was the most galvanizing movement in Native American politics in decades, some tribal leaders said, as Crow, Navajo, Sioux and other traditional rivals united to fight what they considered an assault on their way of life. Cladoosby did not play a high-profile role in the early days of the pipeline controversy. But that day he spoke to a high echelon of power, including Secretary of the Interior Sally Jewell, White House Domestic Policy Council director Cecilia Munoz, and the heads of the Departments of Energy; Agriculture; Education; Health and Human Services; and the Environmental Protection Agency, according to a senior administration official who asked not to be named and to a photo of attendees seen by Reuters. “The world is watching,” he said in prepared remarks shared with Reuters. A few days earlier, video of pipeline security personnel in North Dakota armed with guard dogs and mace trying to disperse protesters went viral on social media. One of the first videos was taken and posted on Facebook by Lonnie Favel, a member of Utah’s Ute tribe who traveled to North Dakota to support the protests. “I was getting messages of support from New Zealand, from Europe, from all over the world,” Favel said. Until then, Obama had not weighed in on the Dakota dispute even though he personally had visited  the Standing Rock in June 2014. Just a day after Cladoosby issued his plea to administration officials, Obama attended a young leaders conference in Laos where a Malaysian woman asked him about the Dakota Access pipeline and how he could ensure a clean water supply and protect ancestral land. Obama said he needed to ask his staff for more information, but touted his track record protecting Native Americans’ “ancestral lands, sacred sites, waters and hunting grounds,” adding, “this is something that I hope will continue as we go forward.” In late 2014, pipeline operator Energy Transfer Partners made a fateful decision. Dallas-based ETP chose to route its proposed Dakota Access pipeline away from North Dakota’s capital, Bismarck, and southward within half a mile of the Standing Rock Sioux tribe’s reservation. Part of its rationale, laid out in a report for the U.S. Army Corps of Engineers, which regulates infrastructure projects that traverse certain inland waterways, was that the route would avoid Bismarck and thus pose no threat to the city’s water supply. The Bismarck route also is more populated and thus would require more easements from multiple landowners. Ironically, that 139-page report concluded the Standing Rock route would raise “no environmental justice issues” because the pipeline would not cross tribal lands. The Army Corps’ decision angered environmental activists and unwittingly introduced a powerful new element into the environmental movement: Indian rights groups, who quickly tapped into an extensive network of green activists forged during five long years of protests against TransCanada’s Keystone XL pipeline, which Obama formally nixed last November. The protest gained steam in February when Standing Rock Sioux leaders asked for legal help from Earthjustice, an environmental law group that had previously helped U.S. tribes and Canadian First Nations fight Kinder Morgan’s Trans Mountain pipeline, according to Jan Hasselman, an attorney from Earthjustice working on the North Dakota case, and tribal leaders. Two months later, about 18 tribe members started praying daily near the pipeline’s planned route in North Dakota. The participants would grow in size, creating a group called the Sacred Stone Camp. The international environmental movement soon took notice, including, 350.org, an environmentalist group that helped defeat the Keystone XL pipeline. In July, the group sent a delegation to the Sacred Stone Camp to see how they could help. In many ways, the Dakota Access pipeline drew its inspiration from the fight to stop the Keystone XL pipeline, according to organizers from 350 and other environmental groups. “We didn’t have to totally reinvent the wheel,” said Josh Nelson of Credo, a progressive advocacy group. By then the Sacred Stone Camp, located alongside the confluence of the Cannon Ball and Missouri rivers about an hour south of Bismarck, had swollen in size to thousands, forming a de facto town of  tents, teepees and trailers, a school, medic, communal kitchen, horse corrals and a legal clinic. The tribal members and environmentalists agreed to seize on the U.S. Army Corps’ “fast-tracking” of permits for the pipeline in late July, which they argued was illegal and a violation of tribal rights, 350.org told Reuters. In this case, the Corps had the right to approve pipelines in general and consider specific local concerns, such as Native issues, if appropriate. The Corps said it effectively considered its due diligence requirement met when it green lit the line in July.  Later that same month, the tribe filed suit against the Army Corps in federal court. While the government’s reversal in September caught most by surprise, a March 29 letter from the Department of the Interior to the Army Corps reviewed by Reuters shows that disagreements within the administration had been percolating for months. The Interior department, which is responsible for protecting Native Americans’ welfare, said the Army Corps “did not adequately justify or otherwise support its conclusion that there would be no significant impacts upon the surrounding environment and community” from the pipeline. Energy Transfer, the Department of Justice, the Army Corps and the Department of the Interior did not respond to requests for comment. The letter presaged the intra-government fighting ahead of the White House’s decision to temporarily block the line. The federal delay of the pipeline “isn’t something that just fell out of the sky,” Archambault, the tribe’s chairman, said in an interview. “We feed (federal regulators) information all the time on everything that’s illegal here.” Archambault declined to discuss responses from federal regulators he received. On September 9, just three days after Cladoosby made his plea at the White House, U.S. District Judge James Boasberg rejected a request from the tribe to block the $3.7 billion project. Minutes after that ruling, the Interior and Justice Departments, along with the Army Corps, suspended construction on a two-mile stretch of federal land below the Missouri River. White House spokesman Josh Earnest said federal regulators, who could still ultimately approve the project, called the pause to make sure the concerns of all parties were taken into account. James Gette, a senior official in the environment and natural resources division of the DOJ, noted in a September 16 hearing that construction was halted mainly because the Dakota Access pipeline didn’t have an easement for the area where the tribe gets its drinking water. Protesters have vowed not to leave their camp until the pipeline is scrapped or moved far away from their reservation. Their concerns about potential spills, it turns out, have precedent. An analysis of government data by Reuters shows that Sunoco Logistics, the future operator of the pipeline and a unit of ETP, has had the highest rate of spills since 2010 than any of its competitors. Sunoco told Reuters it has taken measures to reduce its spill rate. Cladoosby admits he “was really surprised” by the fast moving events after his strategically-timed entreaty. He will be back at the White House on Monday and Tuesday. Leaders of 567 native American tribes will meet with Obama in Washington to tackle a range of issues facing Native Americans from economic development to environmental protection – including the Dakota Access pipeline. (This version of the story corrects the first name of White House Domestic Policy Council director to Cecilia, not Celia in paragraph 10)
Label: Real

# Example 2
Title: Michelle Obama Just BLASTED Conservatives For Their Racist Lies About Our President (VIDEO)
Text: On Saturday, Michelle Obama graced Jackson State University with her presence as she gave the commencement speech to this year s graduating class. Though she normally stays out of the fray with regard to the right s horrible attacks on her husband, the First Lady took a moment to share her thoughts on the insane, racist attacks coming from conservatives in America. As I ve walked this journey with Barack, I ve gotten a pretty good look at what it means to rise above the fray, what it means to set your eyes on the horizon, to devote your life to making things better for those who will come after you,  Obama said, praising the President s ability to  stay the course  no matter what  kind of ugliness is going on at any particular moment. Then she dropped the hammer, condemning the right s use of  hateful and divisive language  directed at her husband: Charges that he doesn t love our country. The time he was called a liar in front of a Joint Session of Congress. The nonstop questions about his birth certificate and his belief in God. She urged the graduating class to follow her husband s example and address such unabashed hatred with dignity: Are you going to get angry or lash out? Or are you going to take a deep breath, straighten your shoulders, lift up your head, and do what Barack Obama has always done  as he says,  When they go low, I go high.' That s the choice Barack and I have made. That s what has kept us sane over the years. We simply do not allow space in our hearts, minds, or souls for darkness,  she said.Barack Obama has encountered an unprecedented amount of mindless vitriol from the stupid part of America, and has always met it with class. But despite that hate, he will be remembered as the best POTUS in this century and last, perhaps ever, because of his dedication to doing the right thing and his refusal to let the haters weigh him down.Featured image via Getty Images/Feng Li
Label: Fake

Now classify: 
Title: "{news_title}"
Text: "{news_text}"
The label of this news is:
"""
)

prompt_template_text = PromptTemplate.from_template("""
You are a news classification assistant. Classify each news as Real or Fake.

Examples:
# Example 1
Text: Three days after guard dogs attacked Native Americans protesting an oil pipeline project in North Dakota in early September, an unprecedented event took place at the White House. Brian Cladoosby, president of the National Congress of American Indians, which represents more than 500 tribes, spoke to nearly a dozen of President Barack Obama’s Cabinet-level advisers at a September 6 meeting of the White House’s three-year-old Native American Affairs Council.  It was the first time a tribal leader addressed a session of the council, and Cladoosby was invited in his role as the Indian Congress’ leader. Cladoosby, a Swinomish Indian from Washington state, spoke twice at the one-hour roundtable. He told Reuters he praised the Obama administration in his opening statement for its track record on Native American issues such as pushing to reform the Indian Health Service. But when Cladoosby gave his closing speech, he delivered an impassioned request to his audience: stand with Native Americans who have united with the Standing Rock Sioux tribe and block construction of the Dakota Access Pipeline, a 1,100 mile conduit to get oil from North Dakota to Illinois.  That plea marked one of the previously unreported turning points in a drama that played out since February and culminated September 9 with an about face by the U.S. government, from giving the pipeline a green light to backing a request from North Dakota’s Standing Rock Sioux to halt construction of the pipeline.  The tribe fears sacred sites could be destroyed during the line’s construction and that a future oil spill would pollute its drinking water. This month’s win for the tribe, which could be reversed by regulators, is a rare instance of protests resulting in quick federal action and the triumph of an unusual alliance between environmentalists and Native Americans, who both say they were emboldened by the defeat of the Keystone XL pipeline last fall. It also was the most galvanizing movement in Native American politics in decades, some tribal leaders said, as Crow, Navajo, Sioux and other traditional rivals united to fight what they considered an assault on their way of life. Cladoosby did not play a high-profile role in the early days of the pipeline controversy. But that day he spoke to a high echelon of power, including Secretary of the Interior Sally Jewell, White House Domestic Policy Council director Cecilia Munoz, and the heads of the Departments of Energy; Agriculture; Education; Health and Human Services; and the Environmental Protection Agency, according to a senior administration official who asked not to be named and to a photo of attendees seen by Reuters. “The world is watching,” he said in prepared remarks shared with Reuters. A few days earlier, video of pipeline security personnel in North Dakota armed with guard dogs and mace trying to disperse protesters went viral on social media. One of the first videos was taken and posted on Facebook by Lonnie Favel, a member of Utah’s Ute tribe who traveled to North Dakota to support the protests. “I was getting messages of support from New Zealand, from Europe, from all over the world,” Favel said. Until then, Obama had not weighed in on the Dakota dispute even though he personally had visited  the Standing Rock in June 2014. Just a day after Cladoosby issued his plea to administration officials, Obama attended a young leaders conference in Laos where a Malaysian woman asked him about the Dakota Access pipeline and how he could ensure a clean water supply and protect ancestral land. Obama said he needed to ask his staff for more information, but touted his track record protecting Native Americans’ “ancestral lands, sacred sites, waters and hunting grounds,” adding, “this is something that I hope will continue as we go forward.” In late 2014, pipeline operator Energy Transfer Partners made a fateful decision. Dallas-based ETP chose to route its proposed Dakota Access pipeline away from North Dakota’s capital, Bismarck, and southward within half a mile of the Standing Rock Sioux tribe’s reservation. Part of its rationale, laid out in a report for the U.S. Army Corps of Engineers, which regulates infrastructure projects that traverse certain inland waterways, was that the route would avoid Bismarck and thus pose no threat to the city’s water supply. The Bismarck route also is more populated and thus would require more easements from multiple landowners. Ironically, that 139-page report concluded the Standing Rock route would raise “no environmental justice issues” because the pipeline would not cross tribal lands. The Army Corps’ decision angered environmental activists and unwittingly introduced a powerful new element into the environmental movement: Indian rights groups, who quickly tapped into an extensive network of green activists forged during five long years of protests against TransCanada’s Keystone XL pipeline, which Obama formally nixed last November. The protest gained steam in February when Standing Rock Sioux leaders asked for legal help from Earthjustice, an environmental law group that had previously helped U.S. tribes and Canadian First Nations fight Kinder Morgan’s Trans Mountain pipeline, according to Jan Hasselman, an attorney from Earthjustice working on the North Dakota case, and tribal leaders. Two months later, about 18 tribe members started praying daily near the pipeline’s planned route in North Dakota. The participants would grow in size, creating a group called the Sacred Stone Camp. The international environmental movement soon took notice, including, 350.org, an environmentalist group that helped defeat the Keystone XL pipeline. In July, the group sent a delegation to the Sacred Stone Camp to see how they could help. In many ways, the Dakota Access pipeline drew its inspiration from the fight to stop the Keystone XL pipeline, according to organizers from 350 and other environmental groups. “We didn’t have to totally reinvent the wheel,” said Josh Nelson of Credo, a progressive advocacy group. By then the Sacred Stone Camp, located alongside the confluence of the Cannon Ball and Missouri rivers about an hour south of Bismarck, had swollen in size to thousands, forming a de facto town of  tents, teepees and trailers, a school, medic, communal kitchen, horse corrals and a legal clinic. The tribal members and environmentalists agreed to seize on the U.S. Army Corps’ “fast-tracking” of permits for the pipeline in late July, which they argued was illegal and a violation of tribal rights, 350.org told Reuters. In this case, the Corps had the right to approve pipelines in general and consider specific local concerns, such as Native issues, if appropriate. The Corps said it effectively considered its due diligence requirement met when it green lit the line in July.  Later that same month, the tribe filed suit against the Army Corps in federal court. While the government’s reversal in September caught most by surprise, a March 29 letter from the Department of the Interior to the Army Corps reviewed by Reuters shows that disagreements within the administration had been percolating for months. The Interior department, which is responsible for protecting Native Americans’ welfare, said the Army Corps “did not adequately justify or otherwise support its conclusion that there would be no significant impacts upon the surrounding environment and community” from the pipeline. Energy Transfer, the Department of Justice, the Army Corps and the Department of the Interior did not respond to requests for comment. The letter presaged the intra-government fighting ahead of the White House’s decision to temporarily block the line. The federal delay of the pipeline “isn’t something that just fell out of the sky,” Archambault, the tribe’s chairman, said in an interview. “We feed (federal regulators) information all the time on everything that’s illegal here.” Archambault declined to discuss responses from federal regulators he received. On September 9, just three days after Cladoosby made his plea at the White House, U.S. District Judge James Boasberg rejected a request from the tribe to block the $3.7 billion project. Minutes after that ruling, the Interior and Justice Departments, along with the Army Corps, suspended construction on a two-mile stretch of federal land below the Missouri River. White House spokesman Josh Earnest said federal regulators, who could still ultimately approve the project, called the pause to make sure the concerns of all parties were taken into account. James Gette, a senior official in the environment and natural resources division of the DOJ, noted in a September 16 hearing that construction was halted mainly because the Dakota Access pipeline didn’t have an easement for the area where the tribe gets its drinking water. Protesters have vowed not to leave their camp until the pipeline is scrapped or moved far away from their reservation. Their concerns about potential spills, it turns out, have precedent. An analysis of government data by Reuters shows that Sunoco Logistics, the future operator of the pipeline and a unit of ETP, has had the highest rate of spills since 2010 than any of its competitors. Sunoco told Reuters it has taken measures to reduce its spill rate. Cladoosby admits he “was really surprised” by the fast moving events after his strategically-timed entreaty. He will be back at the White House on Monday and Tuesday. Leaders of 567 native American tribes will meet with Obama in Washington to tackle a range of issues facing Native Americans from economic development to environmental protection – including the Dakota Access pipeline. (This version of the story corrects the first name of White House Domestic Policy Council director to Cecilia, not Celia in paragraph 10)
Label: Real

# Example 2
Text: On Saturday, Michelle Obama graced Jackson State University with her presence as she gave the commencement speech to this year s graduating class. Though she normally stays out of the fray with regard to the right s horrible attacks on her husband, the First Lady took a moment to share her thoughts on the insane, racist attacks coming from conservatives in America. As I ve walked this journey with Barack, I ve gotten a pretty good look at what it means to rise above the fray, what it means to set your eyes on the horizon, to devote your life to making things better for those who will come after you,  Obama said, praising the President s ability to  stay the course  no matter what  kind of ugliness is going on at any particular moment. Then she dropped the hammer, condemning the right s use of  hateful and divisive language  directed at her husband: Charges that he doesn t love our country. The time he was called a liar in front of a Joint Session of Congress. The nonstop questions about his birth certificate and his belief in God. She urged the graduating class to follow her husband s example and address such unabashed hatred with dignity: Are you going to get angry or lash out? Or are you going to take a deep breath, straighten your shoulders, lift up your head, and do what Barack Obama has always done  as he says,  When they go low, I go high.' That s the choice Barack and I have made. That s what has kept us sane over the years. We simply do not allow space in our hearts, minds, or souls for darkness,  she said.Barack Obama has encountered an unprecedented amount of mindless vitriol from the stupid part of America, and has always met it with class. But despite that hate, he will be remembered as the best POTUS in this century and last, perhaps ever, because of his dedication to doing the right thing and his refusal to let the haters weigh him down.Featured image via Getty Images/Feng Li
Label: Fake

Now classify: 
Text: "{news_text}"
The label of this news is:
"""
)

In [12]:
def is_within_token_limit(tokenizer, prompt_template, max_ctx=4096, news_title="", news_text=""):
    dummy_prompt = prompt_template.format(news_title="", news_text="")
    base_tokens = tokenizer.encode(dummy_prompt, truncation=False)
    title_tokens = tokenizer.encode(news_title, truncation=False)
    text_tokens = tokenizer.encode(news_text, truncation=False)
    return len(base_tokens) + len(title_tokens) + len(text_tokens) <= max_ctx


valid_indices = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    if is_within_token_limit(tokenizer, prompt_template_title.template, max_ctx=4096, news_title=row["title"]) \
        and is_within_token_limit(tokenizer, prompt_template_text.template, max_ctx=4096, news_text=row["text"]) \
        and is_within_token_limit(tokenizer, prompt_template_titleVtext.template, max_ctx=4096, news_title=row["title"], news_text=row["text"]):
        valid_indices.append(idx)


test_df_valid = test_df.loc[valid_indices].reset_index(drop=True)

100%|██████████| 7680/7680 [02:46<00:00, 46.12it/s]


In [13]:
len(test_df_valid)

7617

In [ ]:
test_df_valid.to_csv("filtered_full_test_data.csv", index=False)

In [ ]:
llm = Llama(
    model_path="./models/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf",
    n_ctx=4096,
    n_gpu_layers=32,
    temperature = 0,
    max_tokens = 30,
    verbose=True
)

In [44]:
def classify_news(prompt_template, news_title="", news_text=""):
    if news_title and news_text:
        prompt = prompt_template.format(news_title=news_title, news_text=news_text)
    elif news_title:
        prompt = prompt_template.format(news_title=news_title)
    elif news_text:
        prompt = prompt_template.format(news_text=news_text)
    else:
        return "Missing input"

    response = llm(prompt)
    output_text = response["choices"][0]["text"]

    match = re.search(r"\b(Real|Fake)\b", output_text, re.IGNORECASE)
    return match.group(1) if match else "Unknown"

In [ ]:
options = ["title", "title_and_text", "text"]

for opt in options:
    preds = []
    for _, row in tqdm(test_df_valid.iterrows(), total=len(test_df_valid)):
        if opt == "title":
            pred = classify_news(prompt_template_title, news_title=row["title"])
        elif opt == "title_and_text":
            pred = classify_news(prompt_template_titleVtext, news_title=row["title"], news_text=row["text"])
        else:
            pred = classify_news(prompt_template_text, news_text=row["text"])
        preds.append(pred)

    test_df_valid.loc[:, f"predicted_label_{opt}"] = preds

In [ ]:
test_df_valid.to_csv("filtered_full_test_data_labeled.csv", index=False)

In [ ]:
test_df_valid.head()

,title,text,subject,date,label,date_clean,parsed_date,exclamation_count_title,question_count_title,quotes_count_title,...,noun_ratio_text,verb_ratio_text,adj_ratio_text,sentiment_polarity_text,sentiment_subjectivity_text,capitalization_ratio_text,html_link_count_text,predicted_label_title,predicted_label_title_and_text,predicted_label_text
0,Russia sets out why it thinks U.N. wrongly acc...,Russia on Thursday set out why it disputed U.N...,worldnews,"November 2, 2017",0,"November 2, 2017",2017-11-02,0,0,0,...,0.336986,0.167123,0.082192,-0.074689,0.338302,0.024658,0,Real,Real,Real
1,Scaramucci TV Appearance Goes Off The Rails A...,The most infamous characters from Donald Trump...,worldnews,"September 22, 2017",1,"September 22, 2017",2017-09-22,0,0,0,...,0.275072,0.191977,0.074499,0.065805,0.431691,0.002865,0,Real,Real,Fake
2,Grizzly Miss-Steppe: How Washington Post Rewro...,MEDIA MELTDOWN: Many now believe that Amazon ...,worldnews,"January 9, 2017",1,"January 9, 2017",2017-01-09,0,0,0,...,0.292804,0.167080,0.072787,0.045232,0.419863,0.029777,0,Fake,Real,Fake
3,"SLC Cops Murder Crying Suspect, Shoot Him In ...","On Sunday, August 13th, Salt Lake City Police ...",worldnews,"October 5, 2017",1,"October 5, 2017",2017-10-05,0,0,0,...,0.299228,0.189189,0.044402,0.088066,0.456057,0.030888,0,Fake,Real,Real
4,Trump eyeing Mnuchin and Ross for economy jobs...,Longtime Donald Trump supporter and activist i...,politicsNews,"November 15, 2016",0,"November 15, 2016",2016-11-15,0,0,0,...,0.376171,0.133869,0.087015,0.091284,0.344845,0.010710,0,Fake,Real,Real


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd

# Map original labels: 1 -> Fake, 0 -> Real
test_df_valid['true_label'] = test_df_valid['label'].map({1: 'Fake', 0: 'Real'})

# Prediction column names
prediction_columns = [
    'predicted_label_title',
    'predicted_label_title_and_text',
    'predicted_label_text'
]

# Function to compute metrics while ignoring 'unknown' predictions
def compute_metrics(true, pred):
    filtered = [(t, p) for t, p in zip(true, pred) if p in ['Fake', 'Real']]
    if not filtered:
        return None  # In case all predictions are 'unknown'
    true_filtered, pred_filtered = zip(*filtered)
    
    acc = accuracy_score(true_filtered, pred_filtered)
    prec = precision_score(true_filtered, pred_filtered, pos_label='Fake')
    rec = recall_score(true_filtered, pred_filtered, pos_label='Fake')
    f1 = f1_score(true_filtered, pred_filtered, pos_label='Fake')
    tn, fp, fn, tp = confusion_matrix(true_filtered, pred_filtered, labels=['Real', 'Fake']).ravel()
    fpr = fp / (fp + tn)
    coverage = len(pred_filtered) / len(pred)

    return {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1_score': f1,
        'false_positive_rate': fpr,
        'coverage': coverage
    }

# Compute metrics for each prediction column
results = {}
for col in prediction_columns:
    results[col] = compute_metrics(test_df_valid['true_label'], test_df_valid[col])

# Display results
results_df = pd.DataFrame(results).T
print(results_df)


                                accuracy  precision    recall  f1_score  \
predicted_label_title           0.767647   0.722945  0.779934  0.750359   
predicted_label_title_and_text  0.749094   0.906542  0.475782  0.624045   
predicted_label_text            0.675271   0.820690  0.328025  0.468709   

                                false_positive_rate  coverage  
predicted_label_title                      0.242314  0.982014  
predicted_label_title_and_text             0.038177  0.978469  
predicted_label_text                       0.055556  0.981620  
